# UTA Coding Challenge — step-by-step analysis

Completed scope: workbook inspection, preprocessing helpers, grouped partitions, and a Word progress report. **Model configuration and training are pending.** Run cells in order using Python 3.12. Only x1–x9 are model inputs; ID and predictor_group are metadata. Codex assisted in preparing this code and report. The final submission requires a PDF of at most three pages.

This notebook contains the code needed for the completed steps; no repository clone or local Python modules are required.

In [ ]:
%pip install -q numpy==2.3.5 pandas==2.2.3 scikit-learn==1.8.0 xlrd==2.0.1 python-docx==1.2.0

## 1. Workbook and output folders
In Colab, upload Database.xls when prompted. Outside Colab, set WORKBOOK to its local path. Files in a Colab runtime are temporary; download the output archive before ending the session.

In [ ]:
from pathlib import Path

# Change this path when running outside Colab.
WORKBOOK = Path("Database.xls")
if not WORKBOOK.is_file():
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError("Set WORKBOOK to your Database.xls path.") from exc
    uploaded = files.upload()
    if "Database.xls" not in uploaded:
        raise FileNotFoundError("Please upload the supplied Database.xls workbook.")
    del uploaded

OUTPUT_ROOT = Path("outputs")
for folder in [OUTPUT_ROOT / "preprocessing", OUTPUT_ROOT / "partitions", Path("reports")]:
    folder.mkdir(parents=True, exist_ok=True)

## 2. Inspect and prepare the data
Read the second header row, validate numeric values and IDs, retain every observation, and identify exact repeated predictor combinations. Standardization is defined here but remains unfitted. The initial target representation is log10(y); its inverse yields positive predictions. This is separate from the RMSLE metric definition. Original numerical units must be used in the physics equation.

In [ ]:
"""Inspect the supplied workbook without fitting transforms on held-out data."""
from pathlib import Path
import argparse
import json

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

FEATURES = [f"x{i}" for i in range(1, 10)]


def load_workbook(path):
    # Row 1 contains merged section headings; row 2 contains feature names.
    df = pd.read_excel(path, sheet_name="sorted all", header=1)
    if df.shape[1] != 11 or list(df.columns[1:]) != FEATURES + ["y"]:
        raise ValueError("Unexpected workbook columns; inspect the source header.")
    df.columns = ["ID"] + FEATURES + ["y"]
    df = df.apply(pd.to_numeric, errors="raise")
    if not np.isfinite(df.to_numpy()).all():
        raise ValueError("Missing or nonfinite values require explicit investigation.")
    if df.ID.duplicated().any() or (df.ID % 1 != 0).any():
        raise ValueError("IDs must be unique integers.")
    if (df[["x1", "x2", "y"]] <= 0).any().any():
        raise ValueError("The physical relation and log target require positive x1, x2, y.")
    df["ID"] = df.ID.astype("int64")
    # Exact predictor matches share an ID regardless of their observed target.
    df["predictor_group"] = df.groupby(FEATURES, sort=True, dropna=False).ngroup()
    return df


class FoldPreprocessor:
    """Fit only on a fold's training rows; preserve original units separately.

    Raw features are standardized, including features that can be zero.
    The target is represented as log10(y), with no learned target statistics.
    """

    def __init__(self):
        self.scaler = StandardScaler()

    def fit(self, training_rows):
        self.scaler.fit(training_rows[FEATURES])
        return self

    def transform_inputs(self, rows):
        return self.scaler.transform(rows[FEATURES])

    @staticmethod
    def transform_target(rows):
        values = rows["y"].to_numpy(dtype=float)
        if not np.isfinite(values).all() or (values <= 0).any():
            raise ValueError("Log target requires finite positive values.")
        return np.log10(values)

    @staticmethod
    def inverse_target(log_predictions):
        with np.errstate(over="raise", under="raise", invalid="raise"):
            predictions = np.power(10.0, np.asarray(log_predictions, dtype=float))
        if not np.isfinite(predictions).all() or (predictions <= 0).any():
            raise ValueError("Predictions must be finite and strictly positive.")
        return predictions


def prepare_preprocessing(path, output):
    df = load_workbook(path)
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    groups = df.groupby("predictor_group")
    audit = {
        # Total number of observations.
        "samples": len(df),
        # Names of the nine model input columns.
        "predictors": FEATURES,
        # Total number of missing cells across all columns.
        "missing_cells": int(df.isna().sum().sum()),
        # Total number of NaN or positive/negative infinity cells.
        "nonfinite_cells": int((~np.isfinite(df.to_numpy())).sum()),
        # Number of distinct combinations of all nine input features.
        "unique_predictor_groups": int(groups.ngroups),
        # All rows sharing their feature combination with another row, including the first.
        "rows_in_repeated_predictor_groups": int(df.duplicated(FEATURES, keep=False).sum()),
        # Rows repeating a feature combination after its first occurrence.
        "extra_rows_with_repeated_predictors": int(df.duplicated(FEATURES).sum()),
        # Feature groups containing more than one distinct target value.
        "groups_with_different_targets": int((groups.y.nunique() > 1).sum()),
        # Extra rows with identical features AND target, ignoring sample ID.
        "repeated_records_excluding_id": int(df.duplicated(FEATURES + ["y"]).sum()),
        # No observations are removed by this preprocessing step.
        "removed_rows": 0,
        # Scaling is deferred until training partitions have been constructed.
        "scaling_fitted": False,
    }
    df.to_csv(output / "validated_data.csv", index=False, float_format="%.17g")
    df.describe().to_csv(output / "descriptive_statistics.csv")
    repeated = df[df.duplicated(FEATURES, keep=False)].sort_values(["predictor_group", "ID"])
    repeated.to_csv(output / "repeated_predictors.csv", index=False, float_format="%.17g")
    (output / "data_audit.json").write_text(json.dumps(audit, indent=2) + "\n")
    print(json.dumps(audit, indent=2))




In [ ]:
prepare_preprocessing(WORKBOOK, OUTPUT_ROOT / "preprocessing")
display(load_workbook(WORKBOOK).head())

## 3. Shared grouped partitions
Use seed 42. Hold out 20% of distinct predictor groups, then create five shuffled group folds within training. Keep targets out of partition construction. A validation_fold of k means the row is held out in CV round k; train that round on the other four folds. Test rows have no fold. Reuse the saved assignments for both models. Fit a fresh scaler on fitting rows only in each round; do not fit on validation or test rows.

In [ ]:
"""Save shared, group-disjoint holdout and five-fold assignments (seed 42)."""
import argparse
import json
from pathlib import Path

import pandas as pd
from sklearn.model_selection import GroupKFold, GroupShuffleSplit



SEED = 42


def create_assignments(df):
    # The fraction applies to predictor groups, so row counts may depart from 80/20.
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    train_indices, test_indices = next(splitter.split(df, groups=df.predictor_group))
    assignments = df[["ID", "predictor_group"]].copy()
    assignments["partition"] = "train"
    assignments.loc[test_indices, "partition"] = "test"
    # Fold numbers indicate when a training row is used for validation.
    # Test rows have no validation fold and must never enter cross-validation.
    assignments["validation_fold"] = pd.Series(pd.NA, index=df.index, dtype="Int64")
    training = df.iloc[train_indices]
    folds = GroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    for fold, (_, validation_indices) in enumerate(
        folds.split(training, groups=training.predictor_group), start=1
    ):
        assignments.loc[training.index[validation_indices], "validation_fold"] = fold
    validate_assignments(df, assignments)
    return assignments


def validate_assignments(df, assignments):
    if len(assignments) != len(df) or not assignments.ID.equals(df.ID):
        raise ValueError("Assignments must preserve every sample ID exactly once.")
    if assignments.groupby("predictor_group").partition.nunique().max() != 1:
        raise ValueError("A predictor group crosses the train/test boundary.")
    training = assignments[assignments.partition == "train"]
    testing = assignments[assignments.partition == "test"]
    if training.validation_fold.isna().any() or testing.validation_fold.notna().any():
        raise ValueError("Only training samples must have validation fold assignments.")
    if set(training.validation_fold) != set(range(1, 6)):
        raise ValueError("Exactly five nonempty validation folds are required.")
    if training.groupby("predictor_group").validation_fold.nunique().max() != 1:
        raise ValueError("A predictor group crosses validation folds.")


def prepare_partitions(workbook, output):
    df = load_workbook(workbook)
    assignments = create_assignments(df)
    # Recreate assignments to verify deterministic behavior without using targets.
    pd.testing.assert_frame_equal(assignments, create_assignments(df))
    training = assignments[assignments.partition == "train"]
    testing = assignments[assignments.partition == "test"]
    summary = {
        "seed": SEED,
        "holdout_method": "GroupShuffleSplit(test_size=0.2)",
        "fold_method": "GroupKFold(n_splits=5, shuffle=True)",
        "group_definition": "Exact equality of x1 through x9; target excluded",
        "total_samples": len(df),
        "training_samples": len(training),
        "testing_samples": len(testing),
        "training_fraction": len(training) / len(df),
        "testing_fraction": len(testing) / len(df),
        "training_groups": training.predictor_group.nunique(),
        "testing_groups": testing.predictor_group.nunique(),
        "folds": [
            {"fold": fold,
             "training_samples": int((training.validation_fold != fold).sum()),
             "validation_samples": int((training.validation_fold == fold).sum())}
            for fold in range(1, 6)
        ],
        "checks": {
            "all_samples_assigned_once": True,
            "no_train_test_group_overlap": True,
            "no_cross_fold_group_overlap": True,
            "reproducible_assignments": True,
        },
        "scaling_fitted": False,
        "models_trained": False,
    }
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    assignments.to_csv(output / "partition_assignments.csv", index=False)
    (output / "partition_summary.json").write_text(json.dumps(summary, indent=2) + "\n")
    print(json.dumps(summary, indent=2))




In [ ]:
prepare_partitions(WORKBOOK, OUTPUT_ROOT / "partitions")
display(pd.read_csv(OUTPUT_ROOT / "partitions/partition_assignments.csv").head())

## 4. Update the Word progress report
Regenerate the report for the completed steps. Model results are explicitly pending; this is a draft, not the final PDF submission.

In [ ]:
from docx import Document
from docx.shared import Inches, Pt, RGBColor
from pathlib import Path
import json

root=OUTPUT_ROOT.parent
audit=json.loads((root/'outputs/preprocessing/data_audit.json').read_text())
doc=Document()
sec=doc.sections[0]
sec.top_margin=sec.bottom_margin=Inches(.7)
sec.left_margin=sec.right_margin=Inches(.75)
normal=doc.styles['Normal']
normal.font.name='Calibri'
normal.font.size=Pt(10)
normal.paragraph_format.space_after=Pt(6)
for name in ['Heading 1','Heading 2']:
    doc.styles[name].font.color.rgb=RGBColor.from_string('204864')
doc.add_heading('Data-Driven and Physics-Informed Neural Networks',0)
doc.add_paragraph('Progress report • Data inspection and preprocessing • October 20, 2026',style='Subtitle')
doc.add_paragraph('Status: Initial data inspection is complete and the preprocessing script has run successfully. Grouped train/test and five-fold assignments are complete. Model configuration, training, tuning, evaluation, and figures are pending. This is a working Word draft, not the final submission report.')
doc.add_heading('1. Objective and supplied data',1)
doc.add_paragraph('The challenge compares a feedforward neural network using nine input features with a physics-informed network incorporating y = x₂² / (72.2 × x₁^(−m)). The positive exponent m must be predicted for each sample from its input features and learned jointly during training. No neural network or exponent mapping has been trained at this stage.')
doc.add_paragraph('Database.xls contains one worksheet, “sorted all”. It uses a two-row header: merged section labels in the first row and feature names in the second. Loading with the second row as the header yields 1,865 observations, nine predictors (x1–x9), and one target (y), plus a sample identifier (ID). ID is retained for traceability and excluded from model inputs.')
doc.add_heading('2. Data quality findings',1)
t=doc.add_table(rows=1, cols=2)
t.style='Light Shading Accent 1'
t.rows[0].cells[0].text='Check'
t.rows[0].cells[1].text='Observed result'
for a,b in [
('Observations','1,865'),('Input features','9: x1–x9'),('Missing / nonfinite cells','0 / 0'),('Duplicate sample IDs','0'),('Distinct predictor combinations','1,859'),('Repeated predictor combinations','6 groups, each containing 2 rows'),('Rows in repeated predictor groups','12 (6 additional occurrences)'),('Repeated groups with different y values','4'),('Repeated groups with identical y values','2'),('Rows removed','0')]:
    c=t.add_row().cells;c[0].text=a;c[1].text=b
doc.add_paragraph('Repeated groups are defined by exact equality across all nine predictors, without using y. The two records with identical inputs and target are counted as additional occurrences after the first record; all observations are retained. Different targets for matching inputs indicate that these inputs alone do not uniquely determine the observed target. Their cause has not been established.')
doc.add_heading('3. Numerical ranges',1)
doc.add_paragraph('Observed ranges are x1: 0.2094–0.8100; x2: approximately 0.083098–12,442.844754; and y: approximately 0.000075–46.863905. These three variables are strictly positive. Features x3, x7, and x8 contain zeros. The broad scales support considering input standardization and a logarithmic target representation, without taking logarithms of every input.')
doc.add_page_break()
doc.add_heading('4. Preprocessing implemented',1)
for text in [
'Workbook validation: Require the expected feature columns, numeric finite values, unique integer IDs, and strictly positive x1, x2, and y. Invalid data cause an explicit error rather than silent imputation, clipping, or deletion.',
'Observation retention: Keep all rows, including repeated records and extreme values. No observations have been removed because they may be difficult to predict.',
'Grouping metadata: Assign a predictor_group number to exact matches across x1–x9. This number is only for keeping matching inputs together during later partitioning; it is never a neural-network input.',
'Input scaling: A FoldPreprocessor class provides standardization of the nine raw predictors. Each mean and standard deviation must be learned from the relevant training portion only. The same fitted transform will then be applied to validation data. No scaler has been fitted yet; the grouped partitions are now saved.',
'Target representation: The initial implementation uses log10(y), with inverse transformation 10 raised to the predicted value. This supports positive predictions; numerical overflow, underflow, or invalid outputs are treated as errors rather than silently clipped. This is an initial modeling choice, whose suitability has not yet been evaluated.',
'Physical units: Original feature values remain available. The physical equation must use original x1 and x2 values, rather than standardized inputs. The logarithmic target transform is separate from the required base-10 RMSLE metric, which will be implemented explicitly during evaluation.'
]: doc.add_paragraph(text,style='List Bullet')
doc.add_heading('5. Reproducibility and current outputs',1)
doc.add_paragraph('The repository contains preprocess.py, a pinned requirements.txt, and reproduction instructions in README.md. The preprocessing command has been executed successfully against the supplied workbook. Generated outputs include validated_data.csv, descriptive_statistics.csv, repeated_predictors.csv, and data_audit.json under outputs/preprocessing/. Original observations and IDs are preserved in the validated output.')
doc.add_paragraph('Command: python preprocess.py /path/to/Database.xls')
doc.add_heading('6. Planned next steps and limitations',1)
doc.add_paragraph('A grouped split using seed 42 assigns 1,493 samples (80.05%) to training and 372 (19.95%) to testing. Five training folds contain 298, 298, 300, 300, and 297 validation samples. GroupShuffleSplit uses 20% of predictor groups for testing; shuffled GroupKFold creates training folds. Neither uses targets. Complete sample coverage, no group overlap, and deterministic regeneration were checked. Assignments and a summary are saved under outputs/partitions/. Reuse these assignments for both models. Next, define the data-driven network search space and budget. Fit preprocessing separately within each fold, use validation data for early stopping and selection, and keep test data out of all tuning decisions.')
doc.add_paragraph('Model architectures, search budgets, final training durations, predictive metrics, learned exponent summaries, and required figures are not yet available. No claims about model accuracy or physical agreement can be made. The final submission requires a PDF of at most three pages; this Word draft will be revised and condensed as results become available.')
doc.add_heading('7. Assistance disclosure',1)
doc.add_paragraph('OpenAI Codex assisted with reading the challenge, inspecting the workbook, implementing preprocessing, and drafting this report. The submitter should review, understand, and verify the methods and disclose this assistance in the final submission.')
sec.footer.paragraphs[0].text='UTA Coding Challenge | Working progress report | Preprocessing stage'
(root/'reports').mkdir(parents=True, exist_ok=True)
p=root/'reports/preprocessing_progress_report.docx'
doc.save(p)
# Reopen to confirm the Word package and its expected content are readable.
check=Document(p)
assert len(check.tables)==1
assert any('1,865' in p.text for p in check.paragraphs)
print('Created:',p)
print('Size:',p.stat().st_size,'bytes; Word package reopened successfully')


## 5. Package the outputs
Run this after completing the cells above. It bundles preprocessing outputs, partition assignments, and the Word report. In Colab, the archive will be offered for download. Upload this notebook to GitHub separately when saving your changes.

In [ ]:
import zipfile

archive = Path("challenge_progress.zip")
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for directory in [OUTPUT_ROOT, Path("reports")]:
        for path in sorted(directory.rglob("*")):
            if path.is_file():
                bundle.write(path, arcname=str(path))
print(f"Saved {archive.resolve()}")
try:
    from google.colab import files
except ImportError:
    pass
else:
    files.download(str(archive))

## Next step — not yet implemented
Agree on the data-driven network search ranges, selection criterion, and computational budget before training. No test performance has been examined.